# LeVLJEPA

### model loading

In [ ]:
import timm
import torch
from safetensors.torch import load_file

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create the backbone
levljepa_model = timm.create_model(
    "vit_base_patch16_224",
    pretrained=False,
    num_classes=0,
).to(device)

# Load LeVLJEPA checkpoint
vision_weights = load_file(
    "PATH"
)

missing, unexpected = levljepa_model.load_state_dict(
    {
        k[len("encoder."):]: v
        for k, v in vision_weights.items()
        if k.startswith("encoder.")
    },
    strict=False
)

print("Missing:", missing)
print("Unexpected:", unexpected)

levljepa_model.eval()

# INFONCE

In [ ]:
#
from huggingface_hub import hf_hub_download

config_path = hf_hub_download(
    repo_id="lukaskuhndkfz/InfoNCE-ViT-B-DataComp-200k",
    filename="config.json"
)

print(config_path)

In [ ]:
import json

with open(config_path, "r") as f:
    config = json.load(f)

print(config)

### model loading

In [ ]:
import timm

model_infonce = timm.create_model(
    "vit_base_patch16_224",
    pretrained=False,
    num_classes=0,
)

#print(type(model_infonce))
#print(model_infonce)

In [ ]:
# checkpoint loading into model
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download

vision_weights = load_file(
    hf_hub_download(
        "lukaskuhndkfz/InfoNCE-ViT-B-DataComp-200k",
        "vision_encoder.safetensors"
    )
)

missing, unexpected = model_infonce.load_state_dict(
    {k[len("encoder."):]: v
     for k, v in vision_weights.items()
     if k.startswith("encoder.")},
    strict=False
)

print("Missing:", missing)
print("Unexpected:", unexpected)

# OpenCLIP

In [ ]:
# %pip install open_clip_torch

In [ ]:
import open_clip
model, preprocess, tokenizer = open_clip.create_model_and_transforms(
    "ViT-B-16",
    pretrained="datacomp_l_s1b_b8k"
)
print(type(model))
model = model.to(device)
model.eval()

In [ ]:
#print(model.visual)

In [ ]:
import torch.nn as nn

class OpenCLIPFeatureExtractor(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

openclip_wrapper = OpenCLIPFeatureExtractor(model)

# feature extractor

In [ ]:
# dataset loading
from net2brain.utils.download_datasets import DatasetBonnerPNAS2017, DatasetAlgonauts_NSD_Shared
from pprint import pprint
import os

#paths_bonner = DatasetBonnerPNAS2017.load_dataset()
#pprint(paths_bonner)

paths_NSD_shared  = DatasetAlgonauts_NSD_Shared.load_dataset()
#pprint(paths_NSD_shared)
#paths_NSD_shared.keys()


stimuli_path = paths_NSD_shared["images"]

# count files
# num_images = len([f for f in os.listdir(stimuli_path) if f.endswith(('.jpg', '.png'))])

#print("Number of images:", num_images)

layer_1  → block 1  
...  
layer_11 → block 11   
layer_12 → block 12 → final LayerNorm 

In [ ]:
from torchvision import transforms as T
from torchvision import transforms as trn
import torchextractor as tx
from net2brain.feature_extraction import FeatureExtractor

def my_preprocessor(image, model_name, device):
    """
    Args:
        image (Union[Image.Image, List[Image.Image]]): A PIL Image or a list of PIL Images.
        model_name (str): The name of the model, used to determine specific preprocessing if necessary.
        device (str): The device to which the tensor should be transferred ('cuda' for GPU, 'cpu' for CPU).

    Returns:
        Union[torch.Tensor, List[torch.Tensor]]: The preprocessed image(s) as PyTorch tensor(s).
    """

    transforms = trn.Compose([
        trn.Resize((224, 224)),
        trn.ToTensor(),
        trn.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    img_tensor = transforms(image).unsqueeze(0)
    if device == 'cuda':
        img_tensor = img_tensor.cuda()

    return img_tensor


#-------------
#OpenClip 
#-------------
def my_openclip_transformer_extractor(
    preprocessed_data,
    layers_to_extract,
    model,
):
    visual = model.model.visual
    features = {}

    with torch.no_grad():

        # Patch embedding + CLS + positional embedding + ln_pre
        x = visual._embeds(preprocessed_data)

        for i, block in enumerate(visual.transformer.resblocks):

            x = block(x)

            if i < len(visual.transformer.resblocks) - 1:
                # Intermediate layer: CLS directly after transformer block
                features[f"layer_{i+1}"] = x[:, 0, :]

            else:
                # Final layer: final LayerNorm -> CLS
                x = visual.ln_post(x)
                features[f"layer_{i+1}"] = x[:, 0, :]

    return features
    
#----------
# LeVLJEPA. 
#-----------

def my_levljepa_transformer_extractor(
        preprocessed_data,
        layers_to_extract,
        model
):

    vit = model.model
    features = {}

    with torch.no_grad():

        # Patch embedding
        x = vit.patch_embed(preprocessed_data)

        # Add CLS token
        cls_token = vit.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls_token, x), dim=1)

        # Positional embedding
        x = x + vit.pos_embed
        x = vit.pos_drop(x)

        # Transformer blocks
        for i, block in enumerate(vit.blocks):

            x = block(x)

            if i < len(vit.blocks) - 1:
                # Intermediate layer:
                # use patch tokens, not CLS
                patch_tokens = x[:, 1:, :]
                features[f"layer_{i+1}"] = patch_tokens.mean(dim=1)

            else:
                # Final layer:
                # apply the ViT's final LayerNorm first
                x = vit.norm(x)

                # Use patch tokens for LeVLJEPA dense representation
                patch_tokens = x[:, 1:, :]
                features[f"layer_{i+1}"] = patch_tokens.mean(dim=1)

    return features

def my_infonce_transformer_extractor(
        preprocessed_data,
        layers_to_extract,
        model,
):

    vit = model.model
    features = {}

    with torch.no_grad():

        x = vit.patch_embed(preprocessed_data)

        cls = vit.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls, x), dim=1)

        x = vit.pos_drop(x + vit.pos_embed)

        # Transformer blocks
        for i, block in enumerate(vit.blocks):

            x = block(x)

            # Intermediate layers:
            # CLS directly after the transformer block
            if i < len(vit.blocks) - 1:
                features[f"layer_{i+1}"] = x[:, 0, :]

            # Final layer:
            # transformer block -> final LayerNorm -> CLS
            else:
                x = vit.norm(x)
                features[f"layer_{i+1}"] = x[:, 0, :]

    return features
def my_cleaner(features):
    return features

In [ ]:
import torch.nn as nn

class InfoNCEFeatureExtractor(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

infonce_wrapper = InfoNCEFeatureExtractor(model_infonce).cuda()
levljepa_wrapper = InfoNCEFeatureExtractor(levljepa_model)

In [ ]:
#infonce
from net2brain.feature_extraction import FeatureExtractor
import torch
from PIL import Image
import os



fx = FeatureExtractor(
    model=infonce_wrapper,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    preprocessor=my_preprocessor,
    feature_cleaner=my_cleaner,
    extraction_function=my_infonce_transformer_extractor
)

fx.extract(
    data_path=stimuli_path,
    save_path="PATH",
    consolidate_per_layer=True 
)

In [ ]:
#openclip
fx = FeatureExtractor(
    model=openclip_wrapper,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    preprocessor=my_preprocessor,
    feature_cleaner=my_cleaner,
    extraction_function=my_openclip_transformer_extractor
)

fx.extract(
    data_path=stimuli_path,
    save_path="PATH",
    consolidate_per_layer=True
)

In [ ]:
#levljepa
from net2brain.feature_extraction import FeatureExtractor
import torch
fx = FeatureExtractor(
    model=levljepa_wrapper,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    preprocessor=my_preprocessor,
    feature_cleaner=my_cleaner,
    extraction_function=my_levljepa_transformer_extractor
)

fx.extract(
    data_path=stimuli_path,
    save_path="PATH",
    consolidate_per_layer=True 
)

In [ ]:
#
import inspect

print(inspect.getsource(model.forward))

In [ ]:
from net2brain.feature_extraction import FeatureExtractor

fx = FeatureExtractor(
    model="ViT-L/14",
    netset="Clip",
    device="cpu"
)

print(fx.layers)